# 1. From Python values to trustworthy sensor data

**Start here:** No ML knowledge is assumed. A variable gives a name to a value; a list keeps values in order; a function names a reusable operation. A dictionary associates keys with values. We will use these ideas to describe a sensor window and a dataset split.

**You will learn:** what an array shape means, why subject separation matters, how normalization works, and how a checksum identifies data. Allow roughly 25–40 minutes, including exercises.

**Execution:** Run from the repository or this folder, in order, using the installed project environment. The default lesson needs no downloads, credentials, GPU, or existing model checkpoint. Toy examples are teaching demonstrations, not the real-data study.

**Learning loop:** Read the theory, predict the output, run the example, attempt the exercise, then open the worked answer.


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reports/results.json').is_file())
print('Repository:', ROOT.name)


Repository: Vision


## 1. A window is a table of measurements
A sensor records a value repeatedly. One channel is one measurement axis; one window is a short sequence of readings. In this repository a real input window has nine channels and 128 readings per channel. At 50 Hz, its duration is 128 / 50 = 2.56 seconds.

A batch groups several windows. NumPy stores them in an array whose shape is `(windows, channels, samples)`. Indexing starts at zero. Predict the shape after selecting the first window.


In [2]:
import numpy as np
rng = np.random.default_rng(7)
x = rng.normal(size=(4, 9, 128)).astype('float32')  # Synthetic teaching data.
activities = ['walking', 'upstairs', 'downstairs', 'sitting', 'standing', 'lying']
print('Batch:', x.shape, '| one window:', x[0].shape)
print('Seconds per window:', x.shape[-1] / 50)
assert x[0].shape == (9, 128)


Batch: (4, 9, 128) | one window: (9, 128)
Seconds per window: 2.56


## 2. Split by person before learning statistics
Training examples teach the model. Validation examples help choose settings. Test examples report final performance after those choices. Nearby windows from the same recording share information; randomly scattering them across splits can make evaluation too easy.

A set keeps unique identifiers. Set intersection (`&`) finds shared people. The example below deliberately creates a bad split so that we can detect it without stopping the notebook.


In [3]:
train_people = {1, 2, 3}
val_people = {4}
test_people = {5, 6}
def overlap(a, b):
    return sorted(a & b)
assert not overlap(train_people, val_people)
assert not overlap(train_people, test_people)
bad_validation = {3, 4}
print('Leaked subject IDs:', overlap(train_people, bad_validation))


Leaked subject IDs: [3]


## 3. Normalize with training information only
Channels can have different scales. Subtracting a training mean and dividing by a training standard deviation puts them on comparable scales: `z = (x - mean) / scale`. Fit these statistics once on training data and reuse them on validation and test. Computing a separate mean for the test set would change the evaluation procedure.

`axis=(0, 2)` averages over windows and time while preserving channels. `keepdims=True` retains size-one dimensions so NumPy can broadcast the statistics across the batch.


In [4]:
mean = x.mean(axis=(0, 2), keepdims=True)
scale = np.maximum(x.std(axis=(0, 2), keepdims=True), 1e-6)
z = (x - mean) / scale
future = rng.normal(loc=2, size=(2, 9, 128)).astype('float32')
future_z = (future - mean) / scale
print('Statistics shape:', mean.shape)
print('Training normalized channel means:', np.round(z.mean(axis=(0, 2)), 5))
print('Shifted future mean:', round(float(future_z.mean()), 2))
assert mean.shape == (1, 9, 1)


Statistics shape: (1, 9, 1)
Training normalized channel means: [-0.  0. -0. -0. -0.  0.  0.  0.  0.]
Shifted future mean: 2.03


## 4. Identify the exact evidence
A file name can stay the same even if its contents change. SHA256 maps file bytes to a fingerprint. A matching fingerprint verifies byte identity with a reference, not the scientific correctness of the file.

This is our first use of actual project evidence. It reads a saved manifest without downloading the dataset.


In [5]:
import hashlib
path = ROOT / 'reports/data_manifest.json'
manifest = json.loads(path.read_text())
print('Manifest SHA256:', hashlib.sha256(path.read_bytes()).hexdigest())
print('Manifest fields:', sorted(manifest))


Manifest SHA256: 929566154576e83a773e0ea310030ad7ed651b53d461378af30c0b95d3b53daa
Manifest fields: ['archive_sha256', 'channels', 'dataset', 'doi', 'labels', 'license', 'sample_rate_hz', 'split_policy', 'splits', 'url', 'window_samples']


## Exercise: catch two mistakes
1. Add subject 2 to the test split. Which overlap check catches it?
2. Calculate `future` statistics separately. Why would centering it hide the shift that the example was designed to expose?

<details><summary>Worked answer</summary>

`overlap(train_people, test_people | {2})` returns `[2]`. Test data now shares a training person. Separately centering future data forces its mean toward zero and conceals a change relative to the training reference. Reuse `mean` and `scale` instead.
</details>

**Application bridge:** Inspect `src/edge_agent_lab/data.py` for preparation and split checks, and `models.py` for stored normalization buffers. Continue to [learning and evaluation](02_learning_and_evaluation.ipynb), then use the data section of [Edge Sense](../01_edge_sense.ipynb) to inspect the real protocol.
